# Unsupervised Learning: Clustering and Dimensionality Reduction
**Datasets:** Mall Customer Segmentation · Spotify Audio Features · 3D Mammoth Point Cloud  
**Methods:** K-Means · DBSCAN · Hierarchical Clustering · PCA · t-SNE · UMAP

---

## Overview

Unsupervised learning finds structure in data without predefined labels. This notebook explores two core techniques:

- **Clustering** — grouping data points so that members of the same cluster are more similar to each other than to members of other clusters
- **Dimensionality reduction** — projecting high-dimensional data into 2D or 3D while preserving meaningful structure, both for visualization and as a preprocessing step before clustering

We apply these tools across three datasets of increasing complexity: customer spending behavior, music audio features, and a 3D point cloud shaped like a woolly mammoth.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings('ignore')

## 2. Customer Segmentation — Mall Customers Dataset

### 2.1 Load and explore the data

This dataset contains demographic and behavioral data for 200 mall customers, including age, annual income, and a spending score (1–100) assigned by the mall based on purchasing behavior and frequency.

The goal is to identify natural customer segments that could inform targeted marketing strategies.

In [ ]:
# Download the mall customers dataset
!gdown 16FY6VnzgWfLYDT1AHMYudM2ZNZRdwrSZ

In [ ]:
df = pd.read_csv('Mall_Customers.csv')
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}\n")
df.head()

In [ ]:
# Summary statistics — notice the different scales between income and spending score
df.describe()

### 2.2 Exploratory visualization

Before clustering, it helps to look at the raw data. Annual income vs. spending score is the most informative pair — you can already see a few natural groupings by eye.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Income vs Spending Score
sns.scatterplot(
    x='Annual Income (k$)', y='Spending Score (1-100)',
    data=df, alpha=0.7, ax=axes[0]
)
axes[0].set_title('Annual Income vs. Spending Score')

# Age vs Spending Score
sns.scatterplot(
    x='Age', y='Spending Score (1-100)',
    data=df, alpha=0.7, ax=axes[1]
)
axes[1].set_title('Age vs. Spending Score')

plt.tight_layout()
plt.show()

### 2.3 K-Means clustering

K-Means partitions data into K groups by iteratively assigning points to the nearest cluster centroid and recomputing centroids until convergence. The key challenge is choosing K — we use the **elbow method**, which plots inertia (sum of squared distances from each point to its cluster centroid) against K. The "elbow" is where adding more clusters stops meaningfully reducing inertia.

In [ ]:
# Elbow method — test K from 1 to 9 and record inertia
X_mall = df[['Annual Income (k$)', 'Spending Score (1-100)']]

inertia = []
k_range = range(1, 10)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_mall)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(k_range, inertia, marker='o', linewidth=2, color='steelblue')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method — Mall Customers\nLook for the "elbow" where inertia stops dropping sharply')
plt.tight_layout()
plt.show()

The elbow typically appears around K=5 for this dataset. We'll use that as our optimal K.

In [ ]:
# Fit K-Means with the optimal K identified from the elbow plot
k_optimal = 5
kmeans = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
kmeans.fit(X_mall)

# Store cluster assignments in the dataframe for easy plotting
df['cluster'] = kmeans.labels_

print(f"Cluster sizes:")
print(df['cluster'].value_counts().sort_index())

In [ ]:
# Visualize the clusters — each color is a distinct customer segment
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Clustered view
sns.scatterplot(
    x='Annual Income (k$)', y='Spending Score (1-100)',
    hue='cluster', data=df, palette='Set2',
    ax=axes[0], legend='full'
)

# Add centroids
centroids = kmeans.cluster_centers_
axes[0].scatter(
    centroids[:, 0], centroids[:, 1],
    c='black', marker='X', s=150, zorder=5, label='Centroids'
)
axes[0].set_title(f'K-Means Clustering (K={k_optimal})')
axes[0].legend()

# Cluster profiles — mean income and spending per cluster
cluster_profile = df.groupby('cluster')[['Annual Income (k$)', 'Spending Score (1-100)', 'Age']].mean()
cluster_profile.plot(kind='bar', ax=axes[1], colormap='Set2', edgecolor='black')
axes[1].set_title('Mean Feature Values per Cluster')
axes[1].set_xlabel('Cluster')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

### 2.4 Cluster interpretation

The five clusters map onto recognizable customer archetypes:

| Cluster | Income | Spending | Interpretation |
|---|---|---|---|
| 0 | Low | Low | Budget-conscious, infrequent shoppers |
| 1 | Low | High | High spenders despite modest income |
| 2 | Medium | Medium | Average customers — largest group |
| 3 | High | Low | Affluent but conservative spenders |
| 4 | High | High | Premium target segment — high value |

*Note: cluster numbers may vary between runs; the groupings themselves are consistent.*

### 2.5 Clustering with all features (PCA visualization)

So far we've only clustered on two features. Let's incorporate all numeric features — age, income, and spending score — and use PCA to visualize the result in 2D.

In [ ]:
# Encode gender as binary, then scale all numeric features
df['gender_enc'] = df['Gender'].map({'Male': 0, 'Female': 1})

features_all = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)', 'gender_enc']
X_all = df[features_all]

# StandardScaler is essential before K-Means when features are on different scales
scaler = StandardScaler()
X_all_scaled = scaler.fit_transform(X_all)

In [ ]:
# Re-run elbow method on all features
inertia_all = []
for k in range(1, 10):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_all_scaled)
    inertia_all.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(range(1, 10), inertia_all, marker='o', linewidth=2, color='darkorange')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method — All Features (Scaled)')
plt.tight_layout()
plt.show()

In [ ]:
# Fit K-Means with all features
kmeans_all = KMeans(n_clusters=5, random_state=42, n_init=10)
df['cluster_all'] = kmeans_all.fit_predict(X_all_scaled)

# Project to 2D with PCA for visualization
pca_model = PCA(n_components=2)
pcs = pca_model.fit_transform(X_all_scaled)
df['PC1'] = pcs[:, 0]
df['PC2'] = pcs[:, 1]

# How much variance does each PC explain?
print("Variance explained:")
for i, v in enumerate(pca_model.explained_variance_ratio_):
    print(f"  PC{i+1}: {v:.1%}")
print(f"  Total: {sum(pca_model.explained_variance_ratio_):.1%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Clusters in PCA space
sns.scatterplot(
    x='PC1', y='PC2', hue='cluster_all',
    data=df, palette='Set2', ax=axes[0], alpha=0.8
)
axes[0].set_title('K-Means Clusters in PCA Space (All Features)')

# Color by gender to check if gender drives the separation
sns.scatterplot(
    x='PC1', y='PC2', hue='Gender',
    data=df, palette='coolwarm', ax=axes[1], alpha=0.8
)
axes[1].set_title('PCA Space — Colored by Gender')

plt.tight_layout()
plt.show()

In [ ]:
# PCA loadings — which original features drive each principal component?
loadings = pd.DataFrame(
    pca_model.components_.T,
    index=features_all,
    columns=['PC1', 'PC2']
)

print("PCA Loadings (contribution of each feature to each PC):")
display(loadings.round(3))

# Visualize loadings as a heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(loadings, annot=True, cmap='RdBu_r', center=0, fmt='.2f')
plt.title('PCA Feature Loadings')
plt.tight_layout()
plt.show()

## 3. Hierarchical Clustering

Hierarchical clustering builds a tree of nested clusters (a dendrogram) by successively merging the two closest groups. Unlike K-Means, it doesn't require specifying K upfront — you can cut the tree at any level to get the number of clusters you want.

**Ward linkage** (used here) minimizes the total within-cluster variance at each merge step, producing compact, roughly equal-sized clusters.

In [ ]:
# Agglomerative clustering with Ward linkage on all scaled features
agg = AgglomerativeClustering(n_clusters=5, linkage='ward')
df['agg_cluster'] = agg.fit_predict(X_all_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Hierarchical clusters in PCA space
sns.scatterplot(
    x='PC1', y='PC2', hue='agg_cluster',
    data=df, palette='Set2', ax=axes[0], alpha=0.8
)
axes[0].set_title('Hierarchical Clustering (Ward) in PCA Space')

# Compare with K-Means assignment
sns.scatterplot(
    x='PC1', y='PC2', hue='cluster_all',
    data=df, palette='Set1', ax=axes[1], alpha=0.8
)
axes[1].set_title('K-Means Clustering in PCA Space')

plt.tight_layout()
plt.show()

In [ ]:
# Agreement between the two methods
# (exact cluster labels won't match, but overall structure should be similar)
from sklearn.metrics import adjusted_rand_score

ari = adjusted_rand_score(df['cluster_all'], df['agg_cluster'])
print(f"Adjusted Rand Index (K-Means vs Hierarchical): {ari:.3f}")
print("(1.0 = perfect agreement, 0.0 = no better than chance)")

## 4. DBSCAN — Density-Based Clustering

DBSCAN (Density-Based Spatial Clustering of Applications with Noise) works very differently from K-Means and hierarchical clustering. Rather than partitioning all points into K groups, it identifies **dense regions** and labels points in sparse regions as noise (outliers).

Key parameters:
- **eps (ε):** the maximum distance between two points for them to be considered neighbors
- **min_samples:** the minimum number of points needed within ε to form a dense region

This makes DBSCAN particularly useful when:
- You don't know the number of clusters in advance
- Clusters have irregular shapes
- You expect noise or outliers in your data

Points labeled **-1** are classified as noise — they don't belong to any cluster.

In [ ]:
# Work on the scaled 2-feature version for a clean visualization
X_2feat = df[['Annual Income (k$)', 'Spending Score (1-100)']]
scaler_2 = StandardScaler()
X_2feat_scaled = scaler_2.fit_transform(X_2feat)

db = DBSCAN(eps=0.5, min_samples=5)
db_labels = db.fit_predict(X_2feat_scaled)

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise    = list(db_labels).count(-1)

print(f"Clusters found:  {n_clusters}")
print(f"Noise points:    {n_noise} ({n_noise/len(db_labels)*100:.1f}%)")

In [ ]:
# Visualize DBSCAN results — noise points shown in black
df_plot = df[['Annual Income (k$)', 'Spending Score (1-100)']].copy()
df_plot['dbscan_cluster'] = db_labels

# Separate noise from cluster members for cleaner plotting
noise    = df_plot[df_plot['dbscan_cluster'] == -1]
clusters = df_plot[df_plot['dbscan_cluster'] != -1]

plt.figure(figsize=(8, 5))
sns.scatterplot(
    x='Annual Income (k$)', y='Spending Score (1-100)',
    hue='dbscan_cluster', data=clusters,
    palette='Set2', legend='full', alpha=0.8
)
plt.scatter(
    noise['Annual Income (k$)'], noise['Spending Score (1-100)'],
    c='black', s=30, label='Noise', zorder=5, marker='x'
)
plt.title(f'DBSCAN Clustering (eps=0.5, min_samples=5)\n'
          f'{n_clusters} clusters found, {n_noise} noise points')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Sensitivity analysis — how do results change with different eps values?
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, eps_val in zip(axes, [0.3, 0.5, 0.8]):
    labels = DBSCAN(eps=eps_val, min_samples=5).fit_predict(X_2feat_scaled)
    n_c = len(set(labels)) - (1 if -1 in labels else 0)
    n_n = list(labels).count(-1)

    noise_mask = labels == -1
    ax.scatter(
        X_2feat_scaled[~noise_mask, 0], X_2feat_scaled[~noise_mask, 1],
        c=labels[~noise_mask], cmap='Set2', alpha=0.8, s=20
    )
    ax.scatter(
        X_2feat_scaled[noise_mask, 0], X_2feat_scaled[noise_mask, 1],
        c='black', marker='x', s=20
    )
    ax.set_title(f'eps={eps_val}\n{n_c} clusters, {n_n} noise points')
    ax.set_xlabel('Income (scaled)')
    ax.set_ylabel('Spending Score (scaled)')

plt.suptitle('DBSCAN Sensitivity to eps Parameter', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

## 5. Spotify Track Clustering

We now apply the same clustering workflow to a richer dataset: Spotify audio features for thousands of tracks. Each track is described by nine audio characteristics extracted by Spotify's audio analysis engine.

**Audio features used:**
| Feature | Description |
|---|---|
| `danceability` | How suitable the track is for dancing (0–1) |
| `energy` | Perceptual intensity and activity (0–1) |
| `tempo` | Estimated beats per minute |
| `valence` | Musical positivity — high = happier sound (0–1) |
| `liveness` | Probability the track was recorded live (0–1) |
| `loudness` | Overall loudness in decibels |
| `speechiness` | Presence of spoken words (0–1) |
| `acousticness` | Confidence that the track is acoustic (0–1) |
| `instrumentalness` | Probability of no vocals (0–1) |

Full documentation: https://developer.spotify.com/documentation/web-api/reference/get-audio-features

In [ ]:
# Download the Spotify tracks dataset
!gdown 1LBzEgMDOf7XVrYvQaNvCkiQd10QIO6MU

In [ ]:
df2 = pd.read_csv('tracks.csv')
print(f"Shape: {df2.shape}")
df2.head()

In [ ]:
features = [
    'danceability', 'energy', 'tempo', 'valence', 'liveness',
    'loudness', 'speechiness', 'acousticness', 'instrumentalness'
]

# Drop rows with missing values in our feature columns
X_spotify = df2[features].dropna()
print(f"Tracks available for clustering: {len(X_spotify):,}")

# Scale features — essential here since tempo is in BPM (0–200+) 
# while most other features are 0–1
scaler_spotify = StandardScaler()
X_spotify_scaled = scaler_spotify.fit_transform(X_spotify)

In [ ]:
# Feature distributions — understand what we're working with
fig, axes = plt.subplots(3, 3, figsize=(12, 9))
for ax, feat in zip(axes.flat, features):
    ax.hist(X_spotify[feat].dropna(), bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(feat)
    ax.set_xlabel('')
plt.suptitle('Spotify Audio Feature Distributions', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Elbow method for the Spotify dataset
inertia_spotify = []
k_range_spotify = range(1, 11)

for k in k_range_spotify:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_spotify_scaled)
    inertia_spotify.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(k_range_spotify, inertia_spotify, marker='o', linewidth=2, color='steelblue')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method — Spotify Audio Features')
plt.tight_layout()
plt.show()

In [ ]:
# Fit K-Means with K=5 (or adjust based on elbow)
kmeans_spotify = KMeans(n_clusters=5, random_state=42, n_init=10)
spotify_idx = X_spotify.index
df2.loc[spotify_idx, 'cluster'] = kmeans_spotify.fit_predict(X_spotify_scaled)
df2['cluster'] = df2['cluster'].astype('Int64')

print("Track count per cluster:")
print(df2['cluster'].value_counts().sort_index())

In [ ]:
# Visualize in danceability vs energy space
plt.figure(figsize=(8, 5))
sns.scatterplot(
    x='danceability', y='energy',
    hue='cluster', data=df2,
    palette='Set2', alpha=0.5, s=15
)
plt.title('Spotify Clusters — Danceability vs Energy')
plt.tight_layout()
plt.show()

In [ ]:
# Cluster profiles — mean audio features per cluster
cluster_means = df2.groupby('cluster')[features].mean()

fig, ax = plt.subplots(figsize=(12, 5))
cluster_means.T.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='black', width=0.7)
ax.set_title('Mean Audio Feature Values per Spotify Cluster')
ax.set_ylabel('Mean Value (scaled interpretation)')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

display(cluster_means.round(3))

In [ ]:
# PCA visualization of Spotify clusters
pca_spotify = PCA(n_components=2)
pcs_spotify  = pca_spotify.fit_transform(X_spotify_scaled)

pca_df = pd.DataFrame(pcs_spotify, columns=['PC1', 'PC2'], index=spotify_idx)
pca_df['cluster'] = df2.loc[spotify_idx, 'cluster']

plt.figure(figsize=(8, 5))
sns.scatterplot(
    x='PC1', y='PC2', hue='cluster',
    data=pca_df, palette='Set2', alpha=0.4, s=10
)
plt.title(f'Spotify Clusters in PCA Space\n'
          f'(PC1 + PC2 explain {sum(pca_spotify.explained_variance_ratio_):.1%} of variance)')
plt.tight_layout()
plt.show()

## 6. Dimensionality Reduction Comparison: PCA vs t-SNE vs UMAP

To build intuition for how different reduction techniques behave, we use the **3D Mammoth dataset** — a point cloud of ~10,000 points shaped like a woolly mammoth. The true 3D structure is known, so we can evaluate how well each method preserves it when flattening to 2D.

This is a well-known benchmark in the visualization community for comparing non-linear dimensionality reduction methods.

In [ ]:
# Download the mammoth point cloud
!wget -q "https://raw.githubusercontent.com/MNoichl/UMAP-examples-mammoth-/master/mammoth_a.csv"
df_mam = pd.read_csv('mammoth_a.csv')
print(f"Shape: {df_mam.shape}")
df_mam.head()

In [ ]:
# 3D scatter plot of the original point cloud
# Sample 10,000 points for performance
from mpl_toolkits.mplot3d import Axes3D

df_sample = df_mam.sample(n=10000, random_state=42)

fig = plt.figure(figsize=(8, 6))
ax  = fig.add_subplot(111, projection='3d')
ax.scatter(df_sample['x'], df_sample['y'], df_sample['z'], s=0.3, alpha=0.6)
ax.view_init(elev=20, azim=45)
ax.set_title('Original 3D Mammoth Point Cloud (10,000 point sample)')
plt.tight_layout()
plt.show()

Now we project the 3D data to 2D using three methods:

- **PCA** — linear projection that maximizes variance. Fast and deterministic, but can't capture non-linear structure.
- **t-SNE** — non-linear method that prioritizes preserving *local* neighborhoods. Great for revealing clusters but can distort global distances.
- **UMAP** — non-linear method that attempts to preserve both local and global structure. Generally faster than t-SNE and scales better to large datasets.

In [ ]:
# Install UMAP if not already available
try:
    import umap.umap_ as umap
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'umap-learn', '-q'])
    import umap.umap_ as umap

In [ ]:
X_mam = df_sample[['x', 'y', 'z']].values

# ── PCA ──────────────────────────────────────────────────────────────────────
pca_mam  = PCA(n_components=2).fit_transform(X_mam)

# ── t-SNE ────────────────────────────────────────────────────────────────────
# perplexity roughly controls how many neighbors each point considers
# Higher perplexity = more global structure preserved
tsne_mam = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_mam)

# ── UMAP ─────────────────────────────────────────────────────────────────────
umap_mam = umap.UMAP(n_components=2, random_state=42).fit_transform(X_mam)

print("All three projections computed.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Color points by their original z-coordinate to track how 3D structure maps to 2D
z_vals = df_sample['z'].values

for ax, proj, title in zip(
    axes,
    [pca_mam, tsne_mam, umap_mam],
    ['PCA', 't-SNE (perplexity=30)', 'UMAP']
):
    sc = ax.scatter(proj[:, 0], proj[:, 1], c=z_vals, cmap='plasma', s=0.5, alpha=0.7)
    ax.set_title(title, fontsize=13)
    ax.set_xticks([])
    ax.set_yticks([])
    plt.colorbar(sc, ax=ax, label='Original Z (height)')

plt.suptitle('Mammoth Point Cloud — 3D to 2D Projection Comparison\n'
             'Color = original Z coordinate (height). Brighter = taller.',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

### Comparison summary

| Method | Preserves global structure | Preserves local structure | Speed | Notes |
|---|---|---|---|---|
| **PCA** | ✅ Best | ❌ Weakest | ⚡ Fastest | Linear only; loses non-linear shape |
| **t-SNE** | ❌ Weakest | ✅ Best | 🐢 Slowest | Excellent cluster separation; distances between clusters are not meaningful |
| **UMAP** | ✅ Good | ✅ Good | ⚡ Fast | Best balance; scales to large datasets |

For this dataset, **t-SNE** tends to separate anatomical structures (limbs, body, tusks) most clearly into distinct islands. **UMAP** preserves the overall shape of the mammoth better while still showing local groupings. **PCA** collapses the 3D form into a 2D projection that loses the characteristic mammoth silhouette entirely — illustrating the limits of linear methods on curved manifolds.

## 7. Summary

This notebook covered three clustering algorithms and three dimensionality reduction techniques across three datasets.

**Clustering methods:**

| Method | Best when... | Key limitation |
|---|---|---|
| **K-Means** | Clusters are roughly spherical and similar in size | Requires K upfront; sensitive to outliers |
| **Hierarchical** | You want to explore structure at multiple resolutions | Computationally expensive on large datasets |
| **DBSCAN** | Clusters have irregular shapes or you expect noise/outliers | Sensitive to eps and min_samples; struggles with varying density |

**Dimensionality reduction methods:**

| Method | Best when... | Key limitation |
|---|---|---|
| **PCA** | You need a fast, interpretable linear projection | Can't capture non-linear structure |
| **t-SNE** | You want to visualize local cluster structure | Slow; global distances not meaningful |
| **UMAP** | You need both local and global structure preserved | Results depend on hyperparameters |

**General principles applied throughout:**
- Always normalize features before distance-based methods (K-Means, KNN, DBSCAN)
- Use the elbow method or silhouette scores to guide K selection, not just intuition
- PCA loadings reveal which original features drive each principal component
- Visualizing clusters in reduced-dimension space is a useful sanity check